---
sidebar_label: LangGraph Checkpoint
---


# LangGraph Checkpoint with OceanBase

This notebook shows how to use `OceanBaseCheckpointSaver` as the persistence layer for a LangGraph application.

It covers:
- creating a simple graph with message state
- setting up the OceanBase checkpoint backend
- resuming a thread by `thread_id`
- listing historical checkpoints for time-travel style inspection


## Installation

Install the package together with LangGraph:


In [ ]:
%pip install -qU langchain-oceanbase langgraph

## Start OceanBase

For a local demo, start OceanBase in Docker:


In [ ]:
!docker run --name=oceanbase -e MODE=mini -e OB_SERVER_IP=127.0.0.1 -p 2881:2881 -d oceanbase/oceanbase-ce:latest

## Define the graph state


In [ ]:
from typing import Annotated, TypedDict

from langchain_core.messages import AIMessage, BaseMessage, HumanMessage
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages


class ConversationState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

## Build a simple node and graph

The notebook uses a deterministic echo node so the example is runnable without external LLM credentials.


In [ ]:
def chatbot_node(state: ConversationState) -> dict:
    last_message = state["messages"][-1]
    response_content = f"You said: {last_message.content}"

    if "memory" in last_message.content.lower():
        history = "\n".join(
            f"- {msg.__class__.__name__}: {msg.content}" for msg in state["messages"]
        )
        response_content = f"Conversation history:\n{history}"

    return {"messages": [AIMessage(content=response_content)]}


def build_graph():
    builder = StateGraph(ConversationState)
    builder.add_node("chatbot", chatbot_node)
    builder.add_edge(START, "chatbot")
    builder.add_edge("chatbot", END)
    return builder

## Configure `OceanBaseCheckpointSaver`

Adjust the connection arguments if you are using cloud OceanBase, seekdb, or a different local setup.


In [ ]:
import os

from langchain_oceanbase import OceanBaseCheckpointSaver


connection_args = {
    "host": os.getenv("OCEANBASE_HOST", "127.0.0.1"),
    "port": os.getenv("OCEANBASE_PORT", "2881"),
    "user": os.getenv("OCEANBASE_USER", "root@test"),
    "password": os.getenv("OCEANBASE_PASSWORD", ""),
    "db_name": os.getenv("OCEANBASE_DB", "test"),
}

checkpointer = OceanBaseCheckpointSaver(connection_args=connection_args)
checkpointer.setup()

## Compile the graph with checkpoint persistence


In [ ]:
graph = build_graph().compile(checkpointer=checkpointer)
config = {"configurable": {"thread_id": "checkpoint-demo-thread"}}

## Run a conversation


In [ ]:
for text in [
    "Hello from LangGraph.",
    "Please remember this thread.",
    "Show me the memory.",
]:
    result = graph.invoke({"messages": [HumanMessage(content=text)]}, config)
    print("User:", text)
    print("Agent:", result["messages"][-1].content)
    print("-" * 40)

## Resume state by `thread_id`

This is the core migration path from `ChatMessageHistory`: LangGraph state is stored automatically and organized by thread.


In [ ]:
state_snapshot = graph.get_state(config)
len(state_snapshot.values.get("messages", []))

## Inspect checkpoints

You can iterate through stored checkpoints for debugging, replay, or time-travel workflows.


In [ ]:
for i, checkpoint in enumerate(checkpointer.list(config, limit=5), start=1):
    checkpoint_id = checkpoint.config["configurable"].get("checkpoint_id")
    step = checkpoint.metadata.get("step", "N/A")
    print(f"{i}. checkpoint_id={checkpoint_id} step={step}")

## Next Steps

- See [examples/langgraph_agent.py](../examples/langgraph_agent.py) for the full runnable script.
- See [docs/migration_guide.md](./migration_guide.md) for the migration from `OceanBaseChatMessageHistory` to checkpoint-based LangGraph persistence.
- Use `OceanBaseStore` when you also want namespace-scoped long-term memory alongside graph checkpoints.
